# Manual Test Plan - Interactive Notebook

This notebook executes the manual test plan from `manual-test-plan.md` interactively.

## Usage
- Run the **Setup** cell first to configure environment
- Run individual test sections as needed
- Run the **Generate Report** cell to produce an HTML summary
- Use `Skip` cells to skip sections requiring auth (GitHub)

## 0. Setup - Environment Configuration

In [ ]:
import os
import subprocess
import json
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any

# Test results tracking
test_results: Dict[str, Dict[str, Any]] = {}

# Environment configuration
PR_TEST_ROOT = os.path.expanduser("~/.test-prompt-registry")
XDG_CONFIG_HOME = f"{PR_TEST_ROOT}/xdg"
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PR_BIN = f"node {REPO_ROOT}/lib/dist/cli/main.js"

# Create test directories
os.makedirs(PR_TEST_ROOT, exist_ok=True)
os.makedirs(f"{PR_TEST_ROOT}/xdg", exist_ok=True)
os.makedirs(f"{PR_TEST_ROOT}/project", exist_ok=True)
os.makedirs(f"{PR_TEST_ROOT}/bundles/local-foo", exist_ok=True)

# Set environment variables
os.environ["XDG_CONFIG_HOME"] = XDG_CONFIG_HOME
os.environ["PR_TEST_ROOT"] = PR_TEST_ROOT

print(f"Test root: {PR_TEST_ROOT}")
print(f"CLI binary: {PR_BIN}")
print(f"XDG config: {XDG_CONFIG_HOME}")

## Helper Functions

In [ ]:
def run_cmd(
    cmd: str,
    expected_exit: int = 0,
    capture: bool = True,
    cwd: Optional[str] = None,
    env: Optional[Dict[str, str]] = None
) -> Dict[str, Any]:
    """Run a command and capture output.
    
    Returns:
        dict with keys: exit_code, stdout, stderr, duration
    """
    start = time.time()
    
    # Build environment
    cmd_env = os.environ.copy()
    if env:
        cmd_env.update(env)
    
    # Execute
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=capture,
        text=True,
        cwd=cwd or f"{PR_TEST_ROOT}/project",
        env=cmd_env
    )
    
    duration = time.time() - start
    
    return {
        "exit_code": result.returncode,
        "stdout": result.stdout if capture else "",
        "stderr": result.stderr if capture else "",
        "duration": duration,
        "cmd": cmd
    }

def record_result(
    section: str,
    name: str,
    passed: bool,
    output: str,
    error: Optional[str] = None,
    duration: Optional[float] = None
):
    """Record a test result."""
    if section not in test_results:
        test_results[section] = {
            "name": section,
            "tests": [],
            "start_time": datetime.now().isoformat()
        }
    
    test_results[section]["tests"].append({
        "name": name,
        "passed": passed,
        "output": output,
        "error": error,
        "duration": duration
    })

def assert_exit(result: Dict[str, Any], expected: int = 0) -> bool:
    """Assert exit code matches expected."""
    if result["exit_code"] == expected:
        return True
    return False

def assert_json_field(output: str, field_path: str, expected: Any) -> bool:
    """Assert JSON field matches expected value.
    
    Args:
        output: JSON string
        field_path: Dot-separated path (e.g., 'data.status')
        expected: Expected value
    """
    try:
        data = json.loads(output)
        keys = field_path.split(".")
        value = data
        for key in keys:
            value = value[key]
        return value == expected
    except (json.JSONDecodeError, KeyError, TypeError):
        return False

def assert_json_status(output: str, expected: str = "ok") -> bool:
    """Assert JSON status field matches expected."""
    return assert_json_field(output, "status", expected)

def assert_json_error_code(output: str, expected: str) -> bool:
    """Assert JSON error code matches expected."""
    try:
        data = json.loads(output)
        return data.get("errors", [{}])[0].get("code") == expected
    except (json.JSONDecodeError, IndexError, KeyError):
        return False

def assert_file_exists(path: str) -> bool:
    """Assert file exists."""
    return os.path.exists(os.path.expanduser(path))

def assert_file_contains(path: str, pattern: str) -> bool:
    """Assert file contains pattern."""
    try:
        with open(os.path.expanduser(path), "r") as f:
            content = f.read()
        return pattern in content
    except (FileNotFoundError, IOError):
        return False

print("Helper functions loaded successfully.")

## 1. Unauthenticated Smoke Test

In [ ]:
# 1.1 Version + top-level help
result = run_cmd(f"{PR_BIN} --version")
passed = assert_exit(result, 0) and len(result["stdout"].strip()) > 0
record_result("1", "version", passed, result["stdout"], duration=result["duration"])
print(f"✓ Version check: {'PASS' if passed else 'FAIL'}")

result = run_cmd(f"{PR_BIN} --help")
passed = assert_exit(result, 0) and "prompt-registry" in result["stdout"]
record_result("1", "help", passed, result["stdout"], duration=result["duration"])
print(f"✓ Help check: {'PASS' if passed else 'FAIL'}")

In [ ]:
# 1.2 Empty-state list
env = {"GITHUB_TOKEN": "", "GH_TOKEN": "", "PROMPT_REGISTRY_DISABLE_GH_CLI": "1"}
result = run_cmd(f"{PR_BIN} hub list -o json", env=env)
passed = (
    assert_exit(result) and
    assert_json_status(result["stdout"]) and
    assert_json_field(result["stdout"], "data.hubs", []) and
    assert_json_field(result["stdout"], "data.activeId", None)
)
record_result("1", "empty-state-list", passed, result["stdout"], duration=result["duration"])
print(f"✓ Empty state list: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])
del env

In [ ]:
# 1.3 Unknown top-level command
result = run_cmd(f"{PR_BIN} bogus")
passed = result["exit_code"] != 0
record_result("1", "unknown-command", passed, result["stderr"], duration=result["duration"])
print(f"✓ Unknown command exit code: {'PASS' if passed else 'FAIL'}")

In [ ]:
# 1.4 Index dispatcher: unknown verb exits 64
result = run_cmd(f"{PR_BIN} index ghost")
passed = result["exit_code"] == 64
record_result("1", "index-unknown-verb", passed, result["stderr"], duration=result["duration"])
print(f"✓ Index unknown verb exit 64: {'PASS' if passed else 'FAIL'}")

In [ ]:
# 1.5 `explain` produces non-empty body
result = run_cmd(f"{PR_BIN} explain INDEX.NOT_FOUND")
passed = len(result["stdout"].strip()) > 0
record_result("1", "explain-output", passed, result["stdout"][:200], duration=result["duration"])
print(f"✓ Explain produces output: {'PASS' if passed else 'FAIL'}")

In [ ]:
# 1.6 Schema version stable across output formats
formats = ["text", "json", "yaml", "ndjson"]
all_passed = True
for fmt in formats:
    result = run_cmd(f"{PR_BIN} hub list -o {fmt}")
    passed = assert_exit(result)
    all_passed = all_passed and passed
    record_result("1", f"output-format-{fmt}", passed, "", duration=result["duration"])
    print(f"  -o {fmt}: {'PASS' if passed else 'FAIL'}")

print(f"\n✓ All output formats: {'PASS' if all_passed else 'FAIL'}")

## 2. Initiate User Configuration from Real Hub (SKIP - requires GitHub auth)

In [ ]:
# SKIP: This section requires GitHub authentication
record_result("2", "skip-auth-required", None, "Skipped - requires GitHub auth")
print("Section 2 skipped - requires GitHub authentication")

## 3. Browse Profiles from Hub (SKIP - requires section 2)

In [ ]:
# SKIP: Depends on section 2
record_result("3", "skip-depends-on-section-2", None, "Skipped - depends on section 2")
print("Section 3 skipped - depends on section 2")

## 4. Project-level Configuration (SKIP - target add is a defineCommand)

In [ ]:
# SKIP: target add command is a defineCommand that doesn't work with current clipanion setup
# See manual-test-issues.md for details
record_result("4", "skip-definecommand-limitation", None, "Skipped - target add is a defineCommand (known limitation)")
print("Section 4 skipped - target add is a defineCommand (known limitation)")

## 5. Detached / Default-local Hub Flow

In [ ]:
# 5.1 Add a detached source
result = run_cmd(f"{PR_BIN} source add --type github --url owner/repo --id detached-foo -o json")
passed = assert_json_status(result["stdout"])
record_result("5", "source-add", passed, result["stdout"], duration=result["duration"])
print(f"✓ Source add: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])

In [ ]:
# 5.2 List sources
result = run_cmd(f"{PR_BIN} source list -o text")
passed = assert_exit(result) and "detached-foo" in result["stdout"]
record_result("5", "source-list", passed, result["stdout"], duration=result["duration"])
print(f"✓ Source list: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])

In [ ]:
# 5.3 Remove source
result = run_cmd(f"{PR_BIN} source remove detached-foo -o json")
passed = assert_json_status(result["stdout"])
record_result("5", "source-remove", passed, result["stdout"], duration=result["duration"])
print(f"✓ Source remove: {'PASS' if passed else 'FAIL'}")

## 6. End-to-End Profile Activation (Synthetic Bundle)

In [ ]:
# 6.1 Build synthetic bundle
bundle_dir = f"{PR_TEST_ROOT}/bundles/local-foo"
os.makedirs(f"{bundle_dir}/prompts", exist_ok=True)

# Write deployment manifest
with open(f"{bundle_dir}/deployment-manifest.yml", "w") as f:
    f.write("id: local-foo\nversion: 1.0.0\nname: Local Foo\n")

# Write a prompt
with open(f"{bundle_dir}/prompts/a.md", "w") as f:
    f.write("# A prompt\n")

print(f"✓ Synthetic bundle created at {bundle_dir}")

In [ ]:
# 6.2 Build local hub config
hub_dir = f"{PR_TEST_ROOT}/local-hub"
os.makedirs(hub_dir, exist_ok=True)

hub_config = f"""version: 1.0.0
metadata:
  name: Local Test Hub
  description: synthetic hub for the manual test plan
  maintainer: tester
  updatedAt: '2026-04-26T00:00:00Z'
sources:
  - id: local-foo-src
    name: Local Foo Source
    type: local
    url: {PR_TEST_ROOT}/bundles/local-foo
    enabled: true
    priority: 0
    hubId: local-test-hub
profiles:
  - id: backend
    name: Backend Developer
    bundles:
      - id: local-foo
        version: 1.0.0
        source: local-foo-src
        required: true
"""

with open(f"{hub_dir}/hub-config.yml", "w") as f:
    f.write(hub_config)

print(f"✓ Local hub config created at {hub_dir}")

In [ ]:
# 6.3 Import hub
result = run_cmd(f"{PR_BIN} hub add --type local --location {hub_dir} -o json")
passed = assert_json_status(result["stdout"])
record_result("6", "hub-import", passed, result["stdout"], duration=result["duration"])
print(f"✓ Hub import: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])

In [ ]:
# 6.4 Activate hub
result = run_cmd(f"{PR_BIN} hub use local-test-hub -o json")
passed = (
    assert_json_status(result["stdout"]) and
    assert_json_field(result["stdout"], "data.activeId", "local-test-hub")
)
record_result("6", "hub-activate", passed, result["stdout"], duration=result["duration"])
print(f"✓ Hub activate: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])

In [ ]:
# 6.5 Show profile
result = run_cmd(f"{PR_BIN} profile show backend -o json")
passed = (
    assert_json_status(result["stdout"]) and
    assert_json_field(result["stdout"], "data.profile.id", "backend")
)
record_result("6", "profile-show", passed, result["stdout"], duration=result["duration"])
print(f"✓ Profile show: {'PASS' if passed else 'FAIL'}")
print(result["stdout"])

## 6.6 Profile Activation (SKIP - requires target add)

In [ ]:
# SKIP: Profile activation requires target add first (defineCommand limitation)
record_result("6", "profile-activate-skip", None, "Skipped - requires target add (defineCommand limitation)")
print("Profile activation skipped - requires target add first (defineCommand limitation)")

## 7-27. Additional Test Sections (Placeholder)

Sections 7-27 cover index commands, edge cases, output formats, etc.
These can be added following the same pattern as above.

## Generate Test Report

In [ ]:
def generate_html_report(results: Dict[str, Dict[str, Any]], output_path: str = "test-report.html"):
    """Generate an HTML report from test results."""
    
    total_tests = 0
    passed_tests = 0
    failed_tests = 0
    skipped_tests = 0
    
    html = ["<!DOCTYPE html>"]
    html.append("<html><head>")
    html.append("<title>Manual Test Plan Report</title>")
    html.append("<style>")
    html.append("  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 40px; }")
    html.append("  .summary { background: #f5f5f5; padding: 20px; border-radius: 8px; margin-bottom: 30px; }")
    html.append("  .section { margin-bottom: 30px; border: 1px solid #ddd; border-radius: 8px; overflow: hidden; }")
    html.append("  .section-header { background: #333; color: white; padding: 15px; }")
    html.append("  .test { padding: 15px; border-bottom: 1px solid #eee; }")
    html.append("  .test:last-child { border-bottom: none; }")
    html.append("  .pass { color: green; }")
    html.append("  .fail { color: red; }")
    html.append("  .skip { color: orange; }")
    html.append("  pre { background: #f5f5f5; padding: 10px; border-radius: 4px; overflow-x: auto; }")
    html.append("</style>")
    html.append("</head><body>")
    
    # Summary
    for section in results.values():
        for test in section["tests"]:
            total_tests += 1
            if test["passed"] is True:
                passed_tests += 1
            elif test["passed"] is False:
                failed_tests += 1
            else:
                skipped_tests += 1
    
    html.append("<div class=\"summary\">")
    html.append(f"<h1>Manual Test Plan Report</h1>")
    html.append(f"<p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>")
    html.append(f"<h2>Summary</h2>")
    html.append(f"<p>Total tests: {total_tests}</p>")
    html.append(f"<p class=\"pass\">Passed: {passed_tests}</p>")
    html.append(f"<p class=\"fail\">Failed: {failed_tests}</p>")
    html.append(f"<p class=\"skip\">Skipped: {skipped_tests}</p>")
    html.append(f"<p>Pass rate: {(passed_tests / total_tests * 100) if total_tests > 0 else 0:.1f}%</p>")
    html.append("</div>")
    
    # Sections
    for section_id, section in results.items():
        html.append(f"<div class=\"section\">")
        html.append(f"<div class=\"section-header\"><h2>Section {section_id}: {section['name']}</h2></div>")
        
        for test in section["tests"]:
            status_class = "pass" if test["passed"] is True else ("fail" if test["passed"] is False else "skip")
            status_text = "PASS" if test["passed"] is True else ("FAIL" if test["passed"] is False else "SKIP")
            
            html.append(f"<div class=\"test\">")
            html.append(f"<h3 class=\"{status_class}\">{status_text}: {test['name']}</h3>")
            
            if test["duration"]:
                html.append(f"<p>Duration: {test['duration']:.3f}s</p>")
            
            if test["output"]:
                html.append(f"<p>Output:</p>")
                html.append(f"<pre>{test['output'][:500]}</pre>")
            
            if test["error"]:
                html.append(f"<p class=\"fail\">Error: {test['error']}</p>")
            
            html.append("</div>")
        
        html.append("</div>")
    
    html.append("</body></html>")
    
    report_path = os.path.join(PR_TEST_ROOT, output_path)
    with open(report_path, "w") as f:
        f.write("\n".join(html))
    
    return report_path

report_path = generate_html_report(test_results)
print(f"✓ Report generated: {report_path}")

# Also print summary to console
print("\n=== Test Summary ===")
for section_id, section in test_results.items():
    print(f"\nSection {section_id}:")
    for test in section["tests"]:
        status = "PASS" if test["passed"] is True else ("FAIL" if test["passed"] is False else "SKIP")
        print(f"  {status}: {test['name']}")

## Teardown

In [ ]:
# Clean up test directories (optional - comment out to preserve state)
# import shutil
# if os.path.exists(PR_TEST_ROOT):
#     shutil.rmtree(PR_TEST_ROOT)
#     print(f"Cleaned up: {PR_TEST_ROOT}")
print("Teardown skipped - preserving test state for inspection")